In [1]:
import sys
# append the path of the parent directory
sys.path.append("..")

In [4]:
import math
import os
import time


import numpy as np
np.set_printoptions(legacy='1.25')

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as patches

import seaborn as sns
import time
import json
import pandas as pd
from ctypes import c_int32
from itertools import product
import copy


from tqdm import tqdm

from scipy.stats import pearsonr
from importlib import reload

import orjson
import gzip

from scipy.stats import entropy
from pympler import asizeof

from lib import sketches, visualization_utils, encoders, ploting, pacha_sketch, experiment_utils
reload(ploting)
reload(sketches)
reload(visualization_utils)
reload(encoders)

reload(pacha_sketch)
reload(experiment_utils)

from lib.experiment_utils import add_true_results, compute_true_counts

from lib.sketches import BloomFilter, CountMinSketch, H3HashFunctions, HashFunctionFamily,\
      CountMinSketchHadamard, CountMinSketchLocalHashing, deterministic_hash, simple_deterministic_hash, \
      fast_hash_xx
from lib.visualization_utils import plot_boxplot, plot_relative_error

from lib.encoders import minimal_b_adic_cover, minimal_spatial_b_adic_cover, BAdicCube, BAdicRange, \
      minimal_b_adic_cover_array, downgrade_b_adic_range_indices
from lib.pacha_sketch import PachaSketch, ADTree, BFParameters, CMParameters, \
      cartesian_product, get_n_updates, MaterializedCombinations, get_n_updates_customized

from lib.ploting import set_style, plot_ylabel, plot_legend

from lib import baselines
reload(baselines)

from lib.baselines import CentralDPServer, LDPServer, LDPEncoderGRR, filter_df, query_df, \
      infer_domains_and_ranges, translate_query_region, evaluate_queries, check_accruracy, \
      evaluate_queries_baselines, evaluate_equivalent_pacha_sketches, compute_relative_entropy

# set_style()

In [12]:
def add_true_results_for_dataset(dataset_name, results_folder):
    if dataset_name == "tpch":
        len_df = 599_934
        n_cat = 5
        n_num = 5
    elif dataset_name == "retail":
        len_df = 541_909
        n_cat = 3
        n_num = 3
    elif dataset_name == "census":
        len_df = 377_575
        n_cat = 7
        n_num = 3
    elif dataset_name == "bank":
        len_df = 45_211
        n_cat = 6
        n_num = 4


    path_to_estimates = f"{results_folder}{dataset_name}/{dataset_name}_random.csv"
    path_to_true_counts = f"../results/python/{dataset_name}/{dataset_name}_random.csv"
    add_true_results(path_to_estimates, path_to_true_counts, len_df)

    selectivities = np.array([0.01, 0.02, 0.04, 0.08, 0.16, 0.32, 0.64])
    for sel in selectivities:
        path_to_estimates = f"{results_folder}{dataset_name}/selectivities/{dataset_name}_sel_{sel}.csv"
        path_to_true_counts = f"../results/python/{dataset_name}/selectivities/{dataset_name}_sel_{sel}.csv"
        try:
            add_true_results(path_to_estimates, path_to_true_counts, len_df)
        except Exception as e:
            print(f"Error with path_to_estimates: {path_to_estimates}")
            print(f"Exception: {e}")
            raise

    n_cats = np.arange(1, n_cat+1)
    for n in n_cats:
        path_to_estimates = f"{results_folder}{dataset_name}/categorical/{dataset_name}_cat_{n}.csv"
        path_to_true_counts = f"../results/python/{dataset_name}/categorical/{dataset_name}_cat_{n}.csv"
        add_true_results(path_to_estimates, path_to_true_counts, len_df)

    n_nums = np.arange(1, n_num+1)
    for n in n_nums:
        path_to_estimates = f"{results_folder}{dataset_name}/numerical/{dataset_name}_num_{n}.csv"
        path_to_true_counts = f"../results/python/{dataset_name}/numerical/{dataset_name}_num_{n}.csv"
        add_true_results(path_to_estimates, path_to_true_counts, len_df)

    n_diminant = max(n_cat, n_num)
    n_mix = np.arange(1, n_diminant+1)
    for n in n_mix:
        path_to_estimates = f"{results_folder}{dataset_name}/mixed/{dataset_name}_mix_{n}.csv"
        path_to_true_counts = f"../results/python/{dataset_name}/mixed/{dataset_name}_mix_{n}.csv"
        # add_true_results(path_to_estimates, path_to_true_counts, len_df)
        try:
            add_true_results(path_to_estimates, path_to_true_counts, len_df)
        except Exception as e:
            print(f"Error with path_to_estimates: {path_to_estimates}")
            print(f"Exception: {e}")
            raise

In [14]:
dataset_names = ["tpch", "retail", "census", "bank"]
results_folder = "../results/experiments_results/pacha/"

for dataset_name in dataset_names:
    add_true_results_for_dataset(dataset_name, results_folder)

results_folder = "../results/experiments_results/omni/"

for dataset_name in dataset_names:
    add_true_results_for_dataset(dataset_name, results_folder)

In [5]:
data_base_dir = "../../pacha_experiments/pacha_data/"

data_files = [
    "lineitem_0.1.csv",
    "lineitem_0.5.csv",
    "lineitem_0.5.csv",
    "lineitem_2.csv",
    "lineitem_8.csv",
]

limits = [187_500, 750_000, 3_000_000, 12_000_000, 48_000_000]

results_base_dir = "../results/experiments_results/scalability/"

results_files = [
    "tpchrandom_scale_187_k.csv",
    "tpchrandom_scale_750_k.csv",
    "tpchrandom_scale_3_M.csv",
    "tpchrandom_scale_12_M.csv",
    "tpchrandom_scale_48_M.csv"
]

path_to_queries = "../../PachaSketch/src/main/resources/queries/tpch/tpch_random.json"

for i, results_file in enumerate(results_files):
    compute_true_counts(
        path_to_estimates = f"{results_base_dir}{results_file}",
        path_to_data = f"{data_base_dir}{data_files[i]}",
        path_to_queries = path_to_queries,
        limit = limits[i]
    )

True Count:  62%|██████▏   | 124/200 [04:59<03:47,  3.00s/it]

In [6]:
data_file = "lineitem_0.1.csv"

results_base_dir = "../results/experiments_results/guarantees/"

results_files = [
    "tpchrandom_eps_0.000025_p_0.0025.csv",
    "tpchrandom_eps_0.00005_p_0.005.csv",
    "tpchrandom_eps_0.0001_p_0.01.csv",
    "tpchrandom_eps_0.0002_p_0.02.csv",
    "tpchrandom_eps_0.0004_p_0.04.csv"
]

path_to_queries = "../../PachaSketch/src/main/resources/queries/tpch/tpch_random.json"

for i, results_file in enumerate(results_files):
    compute_true_counts(
        path_to_estimates = f"{results_base_dir}{results_file}",
        path_to_data = f"{data_base_dir}{data_file}",
        path_to_queries = path_to_queries,
    )

True Count: 100%|██████████| 200/200 [00:05<00:00, 36.64it/s]


In [7]:
data_file = "lineitem_0.1.csv"

results_base_dir = "../results/experiments_results/parameters/"

results_files = [
    "bases/tpchrandom_bases_2_2_2_2_2.csv",
    "bases/tpchrandom_bases_5_5_5_5_5.csv",
    "bases/tpchrandom_bases_5_5_5_10_2.csv",
    "bases/tpchrandom_bases_10_10_10_10_10.csv",

    "levels/tpchrandom_levels_3.csv",
    "levels/tpchrandom_levels_5.csv",
    "levels/tpchrandom_levels_7.csv",
    "levels/tpchrandom_levels_9.csv"
]

path_to_queries = "../../PachaSketch/src/main/resources/queries/tpch/tpch_random.json"

for i, results_file in enumerate(results_files):
    compute_true_counts(
        path_to_estimates = f"{results_base_dir}{results_file}",
        path_to_data = f"{data_base_dir}{data_file}",
        path_to_queries = path_to_queries,
    )

True Count: 100%|██████████| 200/200 [00:05<00:00, 36.82it/s]
